# Gold — GA4 marketing tables

Python 3.12 notebook (not Spark). Default lakehouse: **GA4Lakehouse** in workspace **GA4-Report**. Attach that lakehouse (Add data items → From OneLake catalog) before running.

Reads `dbo.bronze_ga4_events` through deltalake (storage bearer token in Fabric; not the raw Files parquet) and DuckDB SQL. Overwrites these `dbo` tables:

- `gold_ga4_sessions` — one row per session (`user_pseudo_id || '-' || ga_session_id`)
- `gold_ga4_daily_kpis` — date × channel_group × device_category
- `gold_ga4_funnel_daily` — session counts reaching each ecommerce step, same grain
- `gold_ga4_items_daily` — date × item_id / name / category / brand
- `gold_ga4_landing_pages` — landing-page attribution with `revenue_per_session` (this is the simpler option: the existing `gold_ga4_page_sessions` table is left unchanged)
- `dim_ga4_page_type` — distinct `page_path` from page_view events, mapped by URL rules

**Session attribution:** `source`, `medium`, and `campaign` are taken from `event_params` on the first event in the session that has them. `session_start` / `first_visit` never carry those keys in this sample, so that is usually a later page_view or engagement event. If a session has no such params, the notebook falls back to `traffic_source.*` (user first-touch, not session source) and records the choice in `attribution_source` (`event_params` or `traffic_source`). `channel_group` is a simple GA4-style default-channel map from the chosen source/medium.

**Landing-page revenue:** a session's purchase revenue is attributed to its landing page (first `page_view`, else first `session_start`). `revenue_per_session` on `gold_ga4_landing_pages` is landing revenue ÷ landing sessions for that date and URL.

**Items:** this sample often attaches a related-item list (~12 rows, `quantity` null) to `view_item` and `add_to_cart`. Those lists are not counted. A view or add is kept when `quantity` is set, or when the event has exactly one item. `begin_checkout` and `purchase` arrays look like real baskets and are fully unnested. `view_item` events with an empty `items` array have no `item_id` and are omitted (the page title is usually the product name).

**Page types:** home, cart (`basket.html`, `shop.axd/cart`), checkout (`yourinfo`, `payment`, `revieworder`, `ordercompleted`, `checkout`), search (`asearch.html`), account (`signin`, `myaccount`, `register`, password, wishlist), product (`.axd` or a Google Merchandise path whose last segment is not a known category), category (`store.html`, `/Google+Redesign/...` category slugs such as Apparel, Mens, Shop+by+Brand), else other (policies, noise). `/Apparel/mens/quickview` follows the parent path.

Physical paths: `/lakehouse/default/Tables/dbo/<table>` (or the equivalent `abfss://` URI).


In [ ]:
# Build marketing gold tables from dbo.bronze_ga4_events (overwrite).
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from deltalake import DeltaTable, write_deltalake

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

ROOT = Path(os.environ.get("LAKEHOUSE_DEFAULT", "/lakehouse/default"))
SOURCE = "bronze_ga4_events"


def _storage():
    """abfss + bearer token in Fabric; local filesystem path otherwise."""
    try:
        import notebookutils
        ctx = notebookutils.runtime.context
        ws = ctx["defaultLakehouseWorkspaceId"]
        lh = ctx["defaultLakehouseId"]
        endpoint = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
        base = f"abfss://{ws}@{endpoint}/{lh}/Tables/dbo"
        opts = {
            "bearer_token": notebookutils.credentials.getToken("storage"),
            "use_fabric_endpoint": "true",
        }
        return base, opts
    except Exception:
        dbo = ROOT / "Tables" / "dbo"
        dbo.mkdir(parents=True, exist_ok=True)
        return str(dbo), None


def _cast_null_type(typ: pa.DataType) -> pa.DataType:
    if pa.types.is_null(typ):
        return pa.string()
    if pa.types.is_struct(typ):
        return pa.struct([
            pa.field(f.name, _cast_null_type(f.type), nullable=f.nullable)
            for f in typ
        ])
    if pa.types.is_list(typ) or pa.types.is_large_list(typ) or pa.types.is_fixed_size_list(typ):
        return pa.list_(_cast_null_type(typ.value_type))
    if pa.types.is_map(typ):
        return pa.map_(_cast_null_type(typ.key_type), _cast_null_type(typ.item_type))
    return typ


def _cast_nulls(table: pa.Table) -> pa.Table:
    new_fields, new_cols = [], []
    for i, field in enumerate(table.schema):
        new_type = _cast_null_type(field.type)
        col = table.column(i)
        if new_type != field.type:
            col = col.cast(new_type)
        new_fields.append(pa.field(field.name, new_type, nullable=field.nullable))
        new_cols.append(col)
    return pa.Table.from_arrays(new_cols, schema=pa.schema(new_fields))


def _arrow(con, sql):
    rel = con.sql(sql)
    if hasattr(rel, "to_arrow_table"):
        return rel.to_arrow_table()
    if hasattr(rel, "fetch_arrow_table"):
        return rel.fetch_arrow_table()
    return rel.arrow()


def _one(con, sql):
    rows = con.sql(sql).fetchall()
    return rows[0] if rows else None


def _write(name, table, base, opts):
    uri = f"{base}/{name}"
    write_deltalake(
        uri,
        _cast_nulls(table),
        mode="overwrite",
        schema_mode="overwrite",
        storage_options=opts,
    )
    return uri



def _struct_field(arr, *names):
    out = arr
    for name in names:
        out = pc.struct_field(out, name)
    return out


def _flatten_batch(batch: pa.RecordBatch) -> pa.Table:
    """Pull device/geo/traffic_source/ecommerce out of nested structs.

    DuckDB 2.x can fail a GROUP BY on the Arrow device struct
    ("struct or union below an offset the conversion would drop"), so gold
    SQL runs on this flat parquet instead of the raw Delta scan.
    """
    device = batch["device"]
    geo = batch["geo"]
    ts = batch["traffic_source"]
    ecom = batch["ecommerce"]
    return pa.table({
        "event_date": batch["event_date"],
        "event_timestamp": batch["event_timestamp"],
        "event_name": batch["event_name"],
        "user_pseudo_id": batch["user_pseudo_id"],
        "event_params": batch["event_params"],
        "items": batch["items"],
        "device_category": _struct_field(device, "category"),
        "operating_system": _struct_field(device, "operating_system"),
        "browser": _struct_field(device, "web_info", "browser"),
        "country": _struct_field(geo, "country"),
        "region": _struct_field(geo, "region"),
        "city": _struct_field(geo, "city"),
        "ts_source": _struct_field(ts, "source"),
        "ts_medium": _struct_field(ts, "medium"),
        "ts_campaign": _struct_field(ts, "name"),
        "purchase_revenue_in_usd": _struct_field(ecom, "purchase_revenue_in_usd"),
        "transaction_id": _struct_field(ecom, "transaction_id"),
    })


def _bronze_flat_parquet(dataset) -> Path:
    tmp = Path(tempfile.mkdtemp(prefix="ga4_flat_")) / "events.parquet"
    writer = None
    try:
        for batch in dataset.to_batches():
            flat = _flatten_batch(batch)
            if writer is None:
                writer = pq.ParquetWriter(str(tmp), flat.schema, compression="zstd")
            writer.write_table(flat)
    finally:
        if writer is not None:
            writer.close()
    return tmp


# Landing-page revenue is a new table built from gold_ga4_sessions
# (landing-page attribution). gold_ga4_page_sessions is left unchanged.
PAGE_PATH_SQL = """
COALESCE(
    NULLIF(
        regexp_replace(
            regexp_replace({col}, '[?#].*$', ''),
            '^[a-zA-Z][a-zA-Z0-9+.-]*://[^/]+',
            ''
        ),
        ''
    ),
    '/'
)"""

# Simple GA4-style default channel group from source/medium.
CHANNEL_SQL = """
CASE
    WHEN lower(COALESCE(session_medium, '')) IN ('cpc', 'ppc', 'paidsearch', 'paid')
      OR (lower(COALESCE(session_medium, '')) IN ('cpm', 'cpv', 'cpp', 'display', 'banner') )
        THEN CASE
            WHEN lower(COALESCE(session_medium, '')) IN ('cpm', 'cpv', 'cpp', 'display', 'banner')
                THEN 'Display'
            ELSE 'Paid Search'
        END
    WHEN lower(COALESCE(session_medium, '')) = 'organic' THEN 'Organic Search'
    WHEN lower(COALESCE(session_medium, '')) = 'referral' THEN 'Referral'
    WHEN lower(COALESCE(session_medium, '')) = 'email' THEN 'Email'
    WHEN lower(COALESCE(session_medium, '')) IN ('affiliate', 'affiliates') THEN 'Affiliates'
    WHEN lower(COALESCE(session_source, '')) = '(direct)'
     AND lower(COALESCE(session_medium, '')) IN ('(none)', '(not set)', '') THEN 'Direct'
    WHEN lower(COALESCE(session_medium, '')) IN ('social', 'social-network', 'social-media', 'sm', 'social_network')
        THEN 'Organic Social'
    ELSE 'Unassigned'
END
"""

# Page-type rules on a normalized path (lower, '+' and '-' -> space,
# collapsed slashes, no trailing slash). last_seg is the last non-quickview
# non-home path segment, so /Apparel/mens/quickview follows Apparel/mens.
CATEGORY_SLUGS = """
'apparel', 'mens', 'womens', 'hats', 'headgear', 'lifestyle', 'drinkware',
'bags', 'small goods', 'clearance', 'new', 'campus collection', 'eco friendly',
'shop by brand', 'youtube', 'google', 'accessories', 'fun', 'stationery',
'office', 'wearables', 'electronics', 'brands', 'notebooks journals',
'writing', 'stickers', 'gift cards', 'kids', 'nest', 'nest usa', 'spring sale',
'sale', 'android', 'google redesign', 'google rmg redesign', 'mens t shirts',
'men s t shirts', 'womens t shirts', 'women s t shirts', 'i am remarkable'
"""
PAGE_TYPE_SQL = f"""
CASE
    WHEN path_norm IS NULL OR path_norm IN ('', '/') OR path_norm IN ('/home', '/home 2', '/home.')
        THEN 'home'
    WHEN path_norm LIKE '%basket%'
      OR path_norm LIKE '%/cart'
      OR path_norm LIKE '%/cart/%'
      OR path_norm LIKE '%shop.axd/cart%'
        THEN 'cart'
    WHEN path_norm LIKE '%yourinfo%'
      OR path_norm LIKE '%payment.html%'
      OR path_norm LIKE '%revieworder%'
      OR path_norm LIKE '%ordercompleted%'
      OR path_norm LIKE '%checkout%'
        THEN 'checkout'
    WHEN path_norm LIKE '%asearch%'
      OR path_norm LIKE '%search%'
        THEN 'search'
    WHEN path_norm LIKE '%signin%'
      OR path_norm LIKE '%myaccount%'
      OR path_norm LIKE '%register%'
      OR path_norm LIKE '%forgot_password%'
      OR path_norm LIKE '%resetpassword%'
      OR path_norm LIKE '%wishlist%'
        THEN 'account'
    WHEN path_norm LIKE '%.axd%'
        THEN 'product'
    WHEN (
        path_norm LIKE '%google%sign%'
        OR path_norm LIKE '/eco/%'
        OR path_norm LIKE '/eco'
    ) AND last_seg NOT IN ({CATEGORY_SLUGS})
        THEN 'product'
    WHEN path_norm LIKE '%google%sign%'
      OR path_norm LIKE '%/store.html%'
      OR path_norm LIKE '/new/%'
      OR path_norm LIKE '/clearance/%'
      OR path_norm LIKE '/eco/%'
      OR path_norm LIKE '/eco'
        THEN 'category'
    ELSE 'other'
END
"""


def main():
    base, opts = _storage()
    src_uri = f"{base}/{SOURCE}"
    print(f"source: {src_uri}")

    bronze_ds = DeltaTable(src_uri, storage_options=opts).to_pyarrow_dataset()
    con = duckdb.connect()
    # Keep a Delta scan for validations against the original ecommerce/items structs.
    con.register("bronze_ga4_events", bronze_ds)
    flat_path = _bronze_flat_parquet(bronze_ds)
    print(f"flat events parquet: {flat_path}")
    try:
        _build(con, base, opts, flat_path)
    finally:
        shutil.rmtree(flat_path.parent, ignore_errors=True)


def _build(con, base, opts, flat_path):

    # Flatten event_params once. In this sample, source/medium/campaign live on
    # event_params (session_start / first_visit never have them), so session
    # attribution takes the first event that has source or medium, and falls
    # back to traffic_source.* (user first-touch) when a session has none.
    con.sql(f"""
    CREATE TEMP TABLE ev AS
    WITH ev0 AS (
        SELECT
            strptime(CAST(event_date AS VARCHAR), '%Y%m%d')::DATE AS event_date,
            event_timestamp,
            event_name,
            user_pseudo_id,
            event_params,
            items,
            device_category,
            operating_system,
            browser,
            country,
            region,
            city,
            ts_source,
            ts_medium,
            ts_campaign,
            purchase_revenue_in_usd,
            transaction_id,
            row_number() OVER () AS _rn
        FROM read_parquet('{flat_path}')
    )
    SELECT
        ev0._rn,
        ev0.event_date,
        ev0.event_timestamp,
        ev0.event_name,
        ev0.user_pseudo_id,
        ev0.device_category,
        ev0.operating_system,
        ev0.browser,
        ev0.country,
        ev0.region,
        ev0.city,
        ev0.ts_source,
        ev0.ts_medium,
        ev0.ts_campaign,
        ev0.purchase_revenue_in_usd,
        ev0.transaction_id,
        ev0.items,
        MAX(CASE WHEN x.key = 'page_location' THEN COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR)) END) AS page_location,
        MAX(CASE WHEN x.key = 'ga_session_id' THEN COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR)) END) AS ga_session_id,
        MAX(CASE WHEN x.key = 'ga_session_number' THEN COALESCE(x.value.int_value, TRY_CAST(x.value.string_value AS BIGINT)) END) AS ga_session_number,
        MAX(CASE WHEN x.key = 'session_engaged' THEN COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR)) END) AS session_engaged,
        MAX(CASE WHEN x.key = 'engagement_time_msec' THEN COALESCE(x.value.int_value, TRY_CAST(x.value.string_value AS BIGINT)) END) AS engagement_time_msec,
        MAX(CASE WHEN x.key = 'source' THEN x.value.string_value END) AS param_source,
        MAX(CASE WHEN x.key = 'medium' THEN x.value.string_value END) AS param_medium,
        MAX(CASE WHEN x.key = 'campaign' THEN x.value.string_value END) AS param_campaign
    FROM ev0
    LEFT JOIN LATERAL UNNEST(ev0.event_params) AS u(x) ON TRUE
    GROUP BY
        ev0._rn, ev0.event_date, ev0.event_timestamp, ev0.event_name, ev0.user_pseudo_id,
        ev0.items, ev0.device_category, ev0.operating_system, ev0.browser,
        ev0.country, ev0.region, ev0.city, ev0.ts_source, ev0.ts_medium, ev0.ts_campaign,
        ev0.purchase_revenue_in_usd, ev0.transaction_id
    """)
    print("flattened events:", con.sql("SELECT COUNT(*) FROM ev").limit(1).fetchall()[0][0])

    sessions = _arrow(con, f"""
    WITH s AS (
        SELECT
            user_pseudo_id,
            ga_session_id,
            user_pseudo_id || '-' || ga_session_id AS session_id,
            min(event_date) AS session_date,
            -- first event that carries source or medium
            arg_min(param_source, CASE WHEN param_source IS NOT NULL OR param_medium IS NOT NULL THEN event_timestamp END) AS param_source,
            arg_min(param_medium, CASE WHEN param_source IS NOT NULL OR param_medium IS NOT NULL THEN event_timestamp END) AS param_medium,
            arg_min(param_campaign, CASE WHEN param_source IS NOT NULL OR param_medium IS NOT NULL OR param_campaign IS NOT NULL THEN event_timestamp END) AS param_campaign,
            arg_min(ts_source, event_timestamp) AS ts_source,
            arg_min(ts_medium, event_timestamp) AS ts_medium,
            arg_min(ts_campaign, event_timestamp) AS ts_campaign,
            COALESCE(
                arg_min(page_location, CASE WHEN event_name = 'page_view' AND page_location IS NOT NULL THEN event_timestamp END),
                arg_min(page_location, CASE WHEN event_name = 'session_start' AND page_location IS NOT NULL THEN event_timestamp END)
            ) AS landing_page,
            arg_min(device_category, event_timestamp) AS device_category,
            arg_min(operating_system, event_timestamp) AS operating_system,
            arg_min(browser, event_timestamp) AS browser,
            arg_min(country, event_timestamp) AS country,
            arg_min(region, event_timestamp) AS region,
            arg_min(city, event_timestamp) AS city,
            BOOL_OR(event_name = 'first_visit') OR BOOL_OR(ga_session_number = 1) AS is_new_user,
            COUNT(*) FILTER (WHERE event_name = 'page_view')::BIGINT AS pageviews,
            (COALESCE(SUM(engagement_time_msec), 0)::DOUBLE / 1000.0) AS engagement_time_sec,
            BOOL_OR(session_engaged = '1') AS engaged,
            COUNT(*) FILTER (WHERE event_name = 'purchase')::BIGINT AS purchases,
            COALESCE(SUM(CASE WHEN event_name = 'purchase' THEN purchase_revenue_in_usd END), 0)::DOUBLE AS revenue,
            COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN COALESCE(transaction_id, CAST(_rn AS VARCHAR)) END)::BIGINT AS transactions,
            BOOL_OR(event_name = 'view_item') AS reached_view_item,
            BOOL_OR(event_name = 'add_to_cart') AS reached_add_to_cart,
            BOOL_OR(event_name = 'begin_checkout') AS reached_begin_checkout,
            BOOL_OR(event_name = 'add_payment_info') AS reached_add_payment_info,
            BOOL_OR(event_name = 'purchase') AS reached_purchase
        FROM ev
        GROUP BY user_pseudo_id, ga_session_id
    ),
    attrib AS (
        SELECT
            session_id,
            session_date,
            user_pseudo_id,
            ga_session_id,
            is_new_user,
            COALESCE(param_source, ts_source) AS session_source,
            COALESCE(param_medium, ts_medium) AS session_medium,
            COALESCE(param_campaign, ts_campaign) AS session_campaign,
            CASE WHEN param_source IS NOT NULL OR param_medium IS NOT NULL THEN 'event_params' ELSE 'traffic_source' END AS attribution_source,
            landing_page,
            device_category,
            operating_system,
            browser,
            country,
            region,
            city,
            pageviews,
            engagement_time_sec,
            engaged,
            purchases,
            revenue,
            transactions,
            reached_view_item,
            reached_add_to_cart,
            reached_begin_checkout,
            reached_add_payment_info,
            reached_purchase
        FROM s
    )
    SELECT
        session_id,
        session_date,
        user_pseudo_id,
        ga_session_id,
        is_new_user,
        session_source,
        session_medium,
        session_campaign,
        attribution_source,
        {CHANNEL_SQL} AS channel_group,
        landing_page,
        {PAGE_PATH_SQL.format(col="landing_page")} AS landing_page_path,
        device_category,
        operating_system,
        browser,
        country,
        region,
        city,
        pageviews,
        engagement_time_sec,
        engaged,
        purchases,
        revenue,
        transactions,
        reached_view_item,
        reached_add_to_cart,
        reached_begin_checkout,
        reached_add_payment_info,
        reached_purchase
    FROM attrib
    """)

    con.register("gold_ga4_sessions", sessions)
    sess_out = sessions.drop([c for c in sessions.column_names if c.startswith("reached_")])
    sess_uri = _write("gold_ga4_sessions", sess_out, base, opts)
    print(f"wrote gold_ga4_sessions: {sess_uri} rows={len(sess_out)}")

    kpis = _arrow(con, """
    SELECT
        session_date,
        channel_group,
        device_category,
        COUNT(*)::BIGINT AS sessions,
        COUNT(DISTINCT user_pseudo_id)::BIGINT AS users,
        COUNT(*) FILTER (WHERE is_new_user)::BIGINT AS new_users,
        COUNT(*) FILTER (WHERE engaged)::BIGINT AS engaged_sessions,
        (COUNT(*) FILTER (WHERE engaged)::DOUBLE / NULLIF(COUNT(*), 0)) AS engagement_rate,
        SUM(purchases)::BIGINT AS purchases,
        SUM(revenue)::DOUBLE AS revenue,
        (COUNT(*) FILTER (WHERE purchases > 0)::DOUBLE / NULLIF(COUNT(*), 0)) AS conversion_rate,
        (SUM(revenue) / NULLIF(SUM(purchases), 0)) AS aov,
        (SUM(revenue) / NULLIF(COUNT(*), 0)) AS revenue_per_session
    FROM gold_ga4_sessions
    GROUP BY session_date, channel_group, device_category
    ORDER BY session_date, channel_group, device_category
    """)
    kpis_uri = _write("gold_ga4_daily_kpis", kpis, base, opts)
    print(f"wrote gold_ga4_daily_kpis: {kpis_uri} rows={len(kpis)}")
    con.register("gold_ga4_daily_kpis", DeltaTable(kpis_uri, storage_options=opts).to_pyarrow_dataset())

    funnel = _arrow(con, """
    SELECT
        session_date,
        channel_group,
        device_category,
        COUNT(*) FILTER (WHERE reached_view_item)::BIGINT AS view_item_sessions,
        COUNT(*) FILTER (WHERE reached_add_to_cart)::BIGINT AS add_to_cart_sessions,
        COUNT(*) FILTER (WHERE reached_begin_checkout)::BIGINT AS begin_checkout_sessions,
        COUNT(*) FILTER (WHERE reached_add_payment_info)::BIGINT AS add_payment_info_sessions,
        COUNT(*) FILTER (WHERE reached_purchase)::BIGINT AS purchase_sessions
    FROM gold_ga4_sessions
    GROUP BY session_date, channel_group, device_category
    ORDER BY session_date, channel_group, device_category
    """)
    funnel_uri = _write("gold_ga4_funnel_daily", funnel, base, opts)
    print(f"wrote gold_ga4_funnel_daily: {funnel_uri} rows={len(funnel)}")
    con.register("gold_ga4_funnel_daily", DeltaTable(funnel_uri, storage_options=opts).to_pyarrow_dataset())

    # Items: this sample often puts a related-item list (~12 rows, quantity NULL)
    # on view_item / add_to_cart. Count a row only when quantity is set, or when
    # the event has exactly one item. Purchase / begin_checkout arrays are real
    # baskets — unnest all of them. view_item events with an empty items array
    # cannot be keyed by item_id and are omitted (page_title is the product name
    # but there is no item_id).
    items = _arrow(con, """
    WITH ev_items AS (
        SELECT
            event_date,
            event_name,
            len(items) AS n_items,
            it.item_id,
            it.item_name,
            it.item_category,
            it.item_brand,
            it.quantity,
            it.item_revenue_in_usd
        FROM ev, UNNEST(items) AS u(it)
        WHERE event_name IN ('view_item', 'add_to_cart', 'begin_checkout', 'purchase')
    ),
    kept AS (
        SELECT *
        FROM ev_items
        WHERE event_name IN ('begin_checkout', 'purchase')
           OR quantity IS NOT NULL
           OR n_items = 1
    )
    SELECT
        event_date,
        item_id,
        item_name,
        item_category,
        item_brand,
        COUNT(*) FILTER (WHERE event_name = 'view_item')::BIGINT AS item_views,
        COUNT(*) FILTER (WHERE event_name = 'add_to_cart')::BIGINT AS add_to_carts,
        COUNT(*) FILTER (WHERE event_name = 'begin_checkout')::BIGINT AS checkouts,
        COALESCE(SUM(CASE WHEN event_name = 'purchase' THEN COALESCE(quantity, 0) END), 0)::BIGINT AS purchases,
        COALESCE(SUM(CASE WHEN event_name = 'purchase' THEN COALESCE(item_revenue_in_usd, 0) END), 0)::DOUBLE AS item_revenue
    FROM kept
    GROUP BY event_date, item_id, item_name, item_category, item_brand
    ORDER BY event_date, item_revenue DESC, item_views DESC, item_id
    """)
    items_uri = _write("gold_ga4_items_daily", items, base, opts)
    print(f"wrote gold_ga4_items_daily: {items_uri} rows={len(items)}")
    con.register("gold_ga4_items_daily", DeltaTable(items_uri, storage_options=opts).to_pyarrow_dataset())

    landing = _arrow(con, f"""
    SELECT
        session_date,
        landing_page,
        ANY_VALUE(landing_page_path) AS landing_page_path,
        COUNT(*)::BIGINT AS sessions,
        COUNT(DISTINCT user_pseudo_id)::BIGINT AS users,
        COUNT(*) FILTER (WHERE engaged)::BIGINT AS engaged_sessions,
        SUM(purchases)::BIGINT AS purchases,
        SUM(revenue)::DOUBLE AS revenue,
        (COUNT(*) FILTER (WHERE purchases > 0)::DOUBLE / NULLIF(COUNT(*), 0)) AS conversion_rate,
        (SUM(revenue) / NULLIF(COUNT(*), 0)) AS revenue_per_session
    FROM gold_ga4_sessions
    WHERE landing_page IS NOT NULL
    GROUP BY session_date, landing_page
    ORDER BY session_date, sessions DESC, landing_page
    """)
    land_uri = _write("gold_ga4_landing_pages", landing, base, opts)
    print(f"wrote gold_ga4_landing_pages: {land_uri} rows={len(landing)}")
    con.register("gold_ga4_landing_pages", DeltaTable(land_uri, storage_options=opts).to_pyarrow_dataset())

    dim = _arrow(con, f"""
    WITH paths AS (
        SELECT DISTINCT {PAGE_PATH_SQL.format(col="page_location")} AS page_path
        FROM ev
        WHERE event_name = 'page_view' AND page_location IS NOT NULL
    ),
    n AS (
        SELECT
            page_path,
            regexp_replace(
                regexp_replace(
                    lower(replace(replace(page_path, '+', ' '), '-', ' ')),
                    '/+',
                    '/',
                    'g'
                ),
                '/+$',
                ''
            ) AS path_norm
        FROM paths
    ),
    segs AS (
        SELECT
            page_path,
            path_norm,
            string_split(trim(path_norm, '/'), '/') AS parts
        FROM n
    ),
    typed AS (
        SELECT
            page_path,
            path_norm,
            CASE
                WHEN len(parts) >= 1 AND parts[len(parts)] IN ('quickview', 'home') AND len(parts) >= 2
                    THEN parts[len(parts) - 1]
                WHEN len(parts) >= 1 THEN parts[len(parts)]
            END AS last_seg
        FROM segs
    )
    SELECT
        page_path,
        {PAGE_TYPE_SQL} AS page_type
    FROM typed
    ORDER BY page_path
    """)
    dim_uri = _write("dim_ga4_page_type", dim, base, opts)
    print(f"wrote dim_ga4_page_type: {dim_uri} rows={len(dim)}")
    con.register("dim_ga4_page_type", DeltaTable(dim_uri, storage_options=opts).to_pyarrow_dataset())

    # ---- validations ----
    bronze_sessions = _one(con, """
    SELECT COUNT(*)::BIGINT
    FROM (
        SELECT DISTINCT
            user_pseudo_id,
            (
                SELECT COALESCE(x.value.string_value, CAST(x.value.int_value AS VARCHAR))
                FROM UNNEST(event_params) AS u(x)
                WHERE x.key = 'ga_session_id'
                LIMIT 1
            ) AS ga_session_id
        FROM bronze_ga4_events
    )
    """)[0]
    gold_sessions, null_dates, gold_rev, attr_params, attr_ts = _one(con, """
    SELECT
        COUNT(*)::BIGINT,
        COUNT(*) FILTER (WHERE session_date IS NULL)::BIGINT,
        COALESCE(SUM(revenue), 0)::DOUBLE,
        COUNT(*) FILTER (WHERE attribution_source = 'event_params')::BIGINT,
        COUNT(*) FILTER (WHERE attribution_source = 'traffic_source')::BIGINT
    FROM gold_ga4_sessions
    """)
    bronze_rev = _one(con, """
    SELECT COALESCE(SUM(ecommerce.purchase_revenue_in_usd), 0)::DOUBLE
    FROM bronze_ga4_events
    WHERE event_name = 'purchase'
    """)[0]
    funnel_co, funnel_pur = _one(con, """
    SELECT
        COALESCE(SUM(begin_checkout_sessions), 0)::BIGINT,
        COALESCE(SUM(purchase_sessions), 0)::BIGINT
    FROM gold_ga4_funnel_daily
    """)
    gold_item_purch_qty, gold_item_rev = _one(con, """
    SELECT
        COALESCE(SUM(purchases), 0)::BIGINT,
        COALESCE(SUM(item_revenue), 0)::DOUBLE
    FROM gold_ga4_items_daily
    """)
    bronze_item_qty, bronze_item_rev = _one(con, """
    SELECT
        COALESCE(SUM(list_sum(list_transform(items, lambda i: COALESCE(i.quantity, 0)))), 0)::BIGINT,
        COALESCE(SUM(list_sum(list_transform(items, lambda i: COALESCE(i.item_revenue_in_usd, 0)))), 0)::DOUBLE
    FROM bronze_ga4_events
    WHERE event_name = 'purchase'
    """)

    print(f"bronze distinct sessions: {bronze_sessions:,}")
    print(f"gold sessions: {gold_sessions:,}")
    print(f"null session_date: {null_dates:,}")
    print(f"attribution event_params: {attr_params:,}  traffic_source fallback: {attr_ts:,}")
    print(f"bronze purchase revenue (ecommerce.purchase_revenue_in_usd): {bronze_rev:,.2f}")
    print(f"gold session revenue: {gold_rev:,.2f}")
    print(f"funnel begin_checkout sessions: {funnel_co:,}")
    print(f"funnel purchase sessions: {funnel_pur:,}")
    print(f"bronze purchase item quantity: {bronze_item_qty:,}")
    print(f"gold items purchases (qty): {gold_item_purch_qty:,}")
    print(f"bronze purchase item_revenue_in_usd: {bronze_item_rev:,.2f}")
    print(f"gold items item_revenue: {gold_item_rev:,.2f}")

    print("channel mix:")
    con.sql("""
    SELECT channel_group, attribution_source, COUNT(*)::BIGINT AS sessions, SUM(revenue)::DOUBLE AS revenue
    FROM gold_ga4_sessions
    GROUP BY 1, 2
    ORDER BY sessions DESC
    """).show()
    print("page_type mix:")
    con.sql("""
    SELECT page_type, COUNT(*)::BIGINT AS pages
    FROM dim_ga4_page_type
    GROUP BY 1
    ORDER BY pages DESC
    """).show()
    print("top landing pages:")
    con.sql("""
    SELECT landing_page_path, SUM(sessions)::BIGINT AS sessions, SUM(revenue)::DOUBLE AS revenue,
           SUM(purchases)::BIGINT AS purchases
    FROM gold_ga4_landing_pages
    GROUP BY 1
    ORDER BY sessions DESC
    LIMIT 8
    """).show()
    print("sample page_type mappings:")
    con.sql("""
    SELECT page_type, page_path
    FROM dim_ga4_page_type
    QUALIFY row_number() OVER (PARTITION BY page_type ORDER BY page_path) <= 3
    ORDER BY page_type, page_path
    """).show()

    errors = []
    if gold_sessions != bronze_sessions:
        errors.append(f"sessions {gold_sessions} != bronze distinct sessions {bronze_sessions}")
    if null_dates:
        errors.append(f"{null_dates} null session_date")
    if abs(gold_rev - bronze_rev) > 0.01:
        errors.append(f"session revenue {gold_rev} != bronze purchase revenue {bronze_rev}")
    if gold_item_purch_qty != bronze_item_qty:
        errors.append(f"item purchases qty {gold_item_purch_qty} != bronze {bronze_item_qty}")
    if abs(gold_item_rev - bronze_item_rev) > 0.01:
        errors.append(f"item revenue {gold_item_rev} != bronze {bronze_item_rev}")

    # Funnel check is informational (not required to be monotonic).
    print(
        f"{'PASS' if funnel_pur <= funnel_co else 'NOTE'} "
        f"gold_ga4_funnel_daily: purchase_sessions={funnel_pur:,} "
        f"begin_checkout_sessions={funnel_co:,}"
    )

    if errors:
        msg = "; ".join(errors)
        print(f"FAIL gold_ga4_marketing: {msg}")
        raise SystemExit(f"FAIL gold_ga4_marketing: {msg}")
    print(f"PASS gold_ga4_sessions: rows={gold_sessions:,} revenue={gold_rev:,.2f}")
    print(f"PASS gold_ga4_daily_kpis: rows={len(kpis):,}")
    print(f"PASS gold_ga4_funnel_daily: rows={len(funnel):,}")
    print(f"PASS gold_ga4_items_daily: rows={len(items):,} purchase_qty={gold_item_purch_qty:,}")
    print(f"PASS gold_ga4_landing_pages: rows={len(landing):,}")
    print(f"PASS dim_ga4_page_type: rows={len(dim):,}")
    print("PASS gold_ga4_marketing")


if __name__ == "__main__":
    main()
